# Sentinel — LoRA fine-tune of LFM2.5-VL on UCF Crime (Kaggle)Kaggle port of `colab_train_lfm_ucf.ipynb`. Same objective: teach a small edgeVLM the surveillance domain so it can answer the tier-2/tier-3 questionsFastVLM-0.5B measurably could not.### Why Kaggle rather than Colab**Save Version → Save & Run All runs headless.** Close the laptop, shut thebrowser — the run continues on Kaggle's machines and the output is waiting.That is exactly the failure that cost this project a training run on2026-08-10. Kaggle also gives ~30 GPU-hours a week against Colab's rationedfree tier, with a 12-hour ceiling per session.### Before you start1. **Settings → Accelerator → GPU T4 x2** (or P100).2. **Settings → Internet → On.** Required for pip and Hugging Face. Kaggle   disables it by default and the failure looks like a confusing timeout.3. Run interactively once to confirm the first cells pass, *then* Save Version   → Save & Run All and walk away.### Do NOT use `odins0n/ucf-crime-dataset`It is the obvious Kaggle hit for "UCF Crime" — 1.38M frames, 11.8 GB, and**every image is 64×64 pixels**. A person in a 64×64 whole-frame thumbnail isabout five pixels across. This project's documented blocker is that 626×360footage already puts people at ~20px, where pose finds two skeletons in 0 of 23frames; training on 64×64 moves in the wrong direction. We use`tanzzpatil/ucf-crime-small` from Hugging Face instead, which keeps usableresolution.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csvimport torch; print('cuda', torch.cuda.is_available())

## 1. Install

In [ ]:
!pip install -q unsloth "transformers>=4.49" accelerate peft trl bitsandbytes datasetsimport unsloth, transformersprint('unsloth', unsloth.__version__, '| transformers', transformers.__version__)

## 2. Clone Sentinel and prepare the datasetThe repo carries `prepare_ucf_dataset.py`, which maps UCF Crime's 14 categoriesonto `ThreatEngine.threat_prompt()`'s output format — a two-line descriptionplus an `INCIDENT:` line — rather than a standalone JSON schema. That is whatmakes a trained checkpoint droppable into the existing pipeline instead ofneeding a new parser written for it.

In [ ]:
!git clone -q https://github.com/louismomo66/security101.git /kaggle/working/sentinel%cd /kaggle/working/sentinel!python -m training.vlm.prepare_ucf_dataset --out /kaggle/working/ucf_prepared

In [ ]:
import json, pathlibd = pathlib.Path('/kaggle/working/ucf_prepared')for f in sorted(d.glob('*.json*'))[:4]:    print(f.name, f.stat().st_size // 1024, 'KB')# Look at one example before training on tens of thousands of them.train = next(d.glob('train*.jsonl'))rows = [json.loads(l) for _, l in zip(range(2), open(train))]for r in rows:    print(json.dumps(r, indent=2)[:600], '\n---')

## 3. Load LFM2.5-VL-1.6B and attach LoRAVision tower frozen — SigLIP already understands imagery, and the gap here islanguage-side: what to *say* about a surveillance frame. Training only thelanguage layers is what keeps this inside a free GPU budget.

In [ ]:
from unsloth import FastVisionModelmodel, tokenizer = FastVisionModel.from_pretrained(    'unsloth/LFM2.5-VL-1.6B', load_in_4bit=True, use_gradient_checkpointing='unsloth')model = FastVisionModel.get_peft_model(    model,    finetune_vision_layers=False,     # frozen    finetune_language_layers=True,    finetune_attention_modules=True,    finetune_mlp_modules=True,    r=16, lora_alpha=16, lora_dropout=0, bias='none', random_state=3407)model.print_trainable_parameters()

## 4. TrainCheckpoints every 200 steps into `/kaggle/working`, which Kaggle persists asthe version's output. The 12-hour session cap means a long run may not finishin one go — `resume_from_checkpoint` picks it up in the next session ratherthan starting from zero.

In [ ]:
import json, pathlibfrom datasets import Datasetfrom trl import SFTTrainer, SFTConfigfrom unsloth import is_bf16_supportedrows = [json.loads(l) for l in open('/kaggle/working/ucf_prepared/train.jsonl')]ds = Dataset.from_list(rows)print(len(ds), 'training examples')FastVisionModel.for_training(model)ckpt = '/kaggle/working/lfm_ucf_ckpt'resume = pathlib.Path(ckpt).exists() and any(pathlib.Path(ckpt).glob('checkpoint-*'))print('resuming from checkpoint:', resume)trainer = SFTTrainer(    model=model, tokenizer=tokenizer, train_dataset=ds,    args=SFTConfig(        per_device_train_batch_size=2, gradient_accumulation_steps=4,        warmup_steps=10, num_train_epochs=1, learning_rate=2e-4,        fp16=not is_bf16_supported(), bf16=is_bf16_supported(),        logging_steps=25, optim='adamw_8bit', weight_decay=0.01,        lr_scheduler_type='linear', seed=3407,        output_dir=ckpt, save_steps=200, save_total_limit=2,        report_to='none', remove_unused_columns=False,        dataset_text_field='', dataset_kwargs={'skip_prepare_dataset': True},        max_seq_length=2048))trainer.train(resume_from_checkpoint=resume)

## 5. Save the adapter

In [ ]:
model.save_pretrained('/kaggle/working/lfm_ucf_lora')tokenizer.save_pretrained('/kaggle/working/lfm_ucf_lora')!du -sh /kaggle/working/lfm_ucf_loraprint('Grab it from the Output tab, or Save Version to keep it attached to this notebook.')

## 6. The number that decides anythingUCF Crime held-out accuracy is **not** that number. The write-up thisreplicates reported 44.8% on a balanced 14-class test set, and a balanced testset says nothing about how often a model fires on ordinary footage — which is99% of what these cameras actually see. That is precisely how a weapon detectoradvertising 93.1% mAP turned out to produce 19.9 false alarms per 100 cleanframes of this project's own video.So the real evaluation runs locally, against Sentinel's own labelled spans —six confirmed incidents and one confirmed normal span:```bashpython -m training.vlm.evaluate --model checkpoints/lfm_ucf_lora```What to look for, in order:1. **How often does it report an incident on the confirmed `normal` span?**   That is the false-alarm rate, and it decides adoption on its own.2. **Does it survive the polarity check?** FastVLM-0.5B answered "yes" to both   a claim and its negation on the same frame. If this one does the same, it is   agreeing with the question rather than reading the image, and the training   bought nothing.3. Only then: does it find the six confirmed incidents?